**INSTALL DEPENDENCIES**

In [9]:
%pip install -q transformers torch tensorflow sentencepiece protobuf tiktoken tf-keras

**IMPORT LIBRARY**

In [ ]:
import os

# HARUS sebelum import tensorflow
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import requests
import torch
import tensorflow as tf

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TFAutoModelForSequenceClassification
)

**CEK INTERNET**

In [3]:
print("="*60)
print("CEK INTERNET")
print("="*60)

try:
    r = requests.get("https://huggingface.co", timeout=10)
    print("Success -  Internet OK")
    print("Status Code:", r.status_code)
except Exception as e:
    print("Failed -  Internet Bermasalah")
    print(e)

**CEK DEVICE**

In [4]:
print("\n" + "="*60)
print("CEK DEVICE")
print("="*60)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch Device:", device)
print("TensorFlow GPU:", len(tf.config.list_physical_devices("GPU")))

**DATA SIMULASI**

In [5]:
texts = [
    "user copied confidential files after working hours",
    "user accessed normal company portal"
]

models = {
    "BERT": "bert-base-uncased",
    "RoBERTa": "roberta-base",
    "DeBERTa": "microsoft/deberta-v3-base"
}

results = {}

**TEST PYTORCH**

In [6]:
print("\n" + "="*60)
print("TEST PYTORCH")
print("="*60)

for model_name, model_id in models.items():
    print(f"\nTesting PyTorch: {model_name}")

    try:
        tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            use_fast=False if model_name == "DeBERTa" else True
        )

        model = AutoModelForSequenceClassification.from_pretrained(
            model_id,
            num_labels=2
        ).to(device)

        inputs = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.argmax(outputs.logits, dim=1)

        results[f"{model_name} - PyTorch"] = predictions.cpu().numpy()
        print("Success")

    except Exception as e:
        results[f"{model_name} - PyTorch"] = "FAILED"
        print("Failed")
        print(e)

**TEST TENSORFLOW**

In [7]:
print("\n" + "="*60)
print("TEST TENSORFLOW")
print("="*60)

for model_name, model_id in models.items():
    print(f"\nTesting TensorFlow: {model_name}")

    try:
        tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            use_fast=False if model_name == "DeBERTa" else True
        )

        model = TFAutoModelForSequenceClassification.from_pretrained(
            model_id,
            num_labels=2
        )

        inputs = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="tf"
        )

        outputs = model(inputs)
        predictions = tf.argmax(outputs.logits, axis=1)

        results[f"{model_name} - TensorFlow"] = predictions.numpy()
        print("Success")

    except Exception as e:
        results[f"{model_name} - TensorFlow"] = "FAILED"
        print("Failed")
        print(e)

**RINGKASAN HASIL AKHIR**

In [8]:
print("\n" + "="*60)
print("RINGKASAN HASIL TEST")
print("="*60)

for name, result in results.items():
    print(f"{name:25} : {result}")

print("="*60)
print("TEST SELESAI")
print("="*60)